# ViFinQA Private V35 - note/ops retrieval + Qwen2.5-Coder-14B code N=5
Independent challenger run over the 282 weak questions with refreshed canonical note and operation routes.

In [ ]:
import glob, json, pathlib

EXPLICIT = pathlib.Path('/kaggle/input/datasets/kien2005/kaggle-payload-private-v35-note-ops-code-n5')
hits = sorted(EXPLICIT.glob('**/retrieval.jsonl')) if EXPLICIT.exists() else []
if not hits:
    hits = [pathlib.Path(p) for p in glob.glob('/kaggle/input/**/retrieval.jsonl', recursive=True) if 'private-v35-note-ops-code-n5' in p]
assert len(hits) == 1, f'Attach exactly one V35 payload: {hits}'
PAYLOAD = str(hits[0].parent)
manifest = json.loads((pathlib.Path(PAYLOAD) / 'payload-manifest.json').read_text())
targets = json.loads((pathlib.Path(PAYLOAD) / 'target_ids.txt').read_text())['ids']
assert manifest['schema_version'] == 2 and len(targets) == 282
runner_text = (pathlib.Path(PAYLOAD) / 'code/kaggle_codegen.py').read_text()
client_text = (pathlib.Path(PAYLOAD) / 'code/vifinqa/codegen/llm_client.py').read_text()
assert 'private-n5-safe-v4' in runner_text and 'truncation_side = "left"' in client_text, 'STALE DATASET: upload a new V35 payload version before running'
print('PAYLOAD', PAYLOAD, '| targets', len(targets), '| files', len(manifest['files']))

In [ ]:
import pathlib, shutil, torch
print('GPUs', torch.cuda.device_count(), torch.cuda.get_device_name(0))
SRC, DST = pathlib.Path(PAYLOAD) / 'code', pathlib.Path('/kaggle/working/code')
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print('verified payload code copied to', DST)

In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"

In [ ]:
import collections, json, os, pathlib, subprocess, sys, time
out = pathlib.Path('/kaggle/working/private_v35_note_ops_qwen14b_code_n5_raw.jsonl')
cmd = [sys.executable, '/kaggle/working/code/kaggle_codegen.py',
       '--payload', PAYLOAD, '--backend', 'hf',
       '--model', 'Qwen/Qwen2.5-Coder-14B-Instruct', '--load-4bit',
       '--llm-mode', 'code', '--llm-target', 'all', '--no-rule-fallback',
       '--debug-rounds', '0', '--llm-ids-file', f'{PAYLOAD}/target_ids.txt',
       '--out', str(out), '--n', '5', '--temperature', '0.50', '--k', '15',
       '--max-tokens', '256', '--batch-size', '1', '--max-input-tokens', '3600',
       '--checkpoint-every', '8', '--time-budget-min', '510', '--seed', '97',
       '--smoke-first', '3']
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
started = time.time()
log_path = out.with_suffix('.runner.log')
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code:
    print('RUNNER LOG SAVED:', log_path)
    print('RUNNER LOG TAIL:\n' + ''.join(log_path.read_text(errors='replace').splitlines(keepends=True)[-80:]))
    for diagnostic_path in [out.with_name(f'{out.stem}.smoke{out.suffix}'), out]:
        print('DIAGNOSTIC FILE', diagnostic_path, 'exists=', diagnostic_path.exists())
        if diagnostic_path.exists():
            diagnostic_rows = [json.loads(line) for line in diagnostic_path.open()]
            selected = [r for r in diagnostic_rows if r['id'] in set(targets)]
            print(collections.Counter(r.get('source') for r in selected))
            print([(r['id'], r.get('llm_diagnostics', {})) for r in selected[:3]])
    raise RuntimeError(f'runner failed with exit code {return_code}; see {log_path}')
print(f'completed in {(time.time() - started) / 60:.1f} min -> {out}')

In [ ]:
import collections, json, math, pathlib
out = pathlib.Path('/kaggle/working/private_v35_note_ops_qwen14b_code_n5_raw.jsonl')
rows = [json.loads(line) for line in out.open(encoding='utf-8')]
assert len(rows) == 1012 and len({r['id'] for r in rows}) == 1012
llm = [r for r in rows if str(r.get('source', '')).startswith('llm')]
assert llm, 'Full run produced zero LLM candidates; inspect the smoke output/logs'
assert {r['id'] for r in llm} <= set(targets)
assert all(math.isfinite(float(r['answer'])) for r in rows)
print('sources', collections.Counter(r['source'] for r in rows))
print('LLM votes', collections.Counter(r.get('votes', 0) for r in llm))
print('strict 4/5 code candidates', sum(int(r.get('votes', 0)) >= 4 for r in llm))
print('DOWNLOAD', out)